<a href="https://colab.research.google.com/github/Rosary-mom/Bio2.0/blob/main/k3/LOGOS_Cockpit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LOGOS-Cockpit · Academical Research Library / UniCampus
**Magnifica Humanitas · ROSARY** — theologische Recherche mit Provenienz  
*Uwe Rosenkranz · EKIR / EAIR · Colab Free / T4*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rosary-mom/Bio2.0/blob/main/k3/LOGOS_Cockpit.ipynb)

### So startest du (4 Schritte)
1. **Laufzeit → Laufzeittyp ändern → T4-GPU** (LLM optional; Retrieval läuft auch ohne GPU).
2. **Laufzeit → Alle ausführen** (von oben nach unten).
3. Corpus liegt unter `logos_corpus/` (eigene/freie Texte). Du kannst später weitere `.md`/`.txt` ablegen.
4. Demo-Frage stellen oder Slider anpassen — Antworten nur aus Treffern mit Beleg.

**Leitsatz:** *Verstehen schlägt bloße Größe.*  
**Seestern:** *Logos Max bleibt das menschliche Werkzeug; dieses Cockpit nutzt nur eigenes/freies Material.*  
**Logos-/Corpus-Regel:** Eigene, von Uwe in Logos erstellte Inhalte (Notizen, Predigten, Auslegungen) sind zulässig per manuellem Export; Volltexte aus lizenzierten Logos-Bibliotheksressourcen nur als kurze Zitate mit Quellenangabe. Quellen [1]–[4] im Blog sind Zitationsnachweise, keine Library-Volltexte.

## 0 · Abhängigkeiten & Setup

In [ ]:
import sys, subprocess, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

need = []
for mod, pkg in [
    ("ipywidgets", "ipywidgets"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
]:
    try:
        __import__(mod)
    except Exception:
        need.append(pkg)

# Embeddings optional (Netz/GPU); fehlen → BM25-Fallback
try:
    import sentence_transformers  # noqa: F401
except Exception:
    # nicht zwingend installieren im Headless-Smoke; nur wenn erlaubt
    if os.environ.get("LOGOS_INSTALL_EMB", "0").strip() in {"1", "true", "yes"}:
        need.append("sentence-transformers")

# LLM optional
for mod, pkg in [("transformers", "transformers"), ("accelerate", "accelerate"), ("bitsandbytes", "bitsandbytes")]:
    try:
        __import__(mod)
    except Exception:
        if os.environ.get("LOGOS_INSTALL_LLM", "0").strip() in {"1", "true", "yes"} or not os.environ.get("SKIP_LLM", "0").startswith("1"):
            # In Colab mit LLM: installieren; Smoke mit SKIP_LLM=1: überspringen
            if os.environ.get("SKIP_LLM", "0").strip().lower() not in {"1", "true", "yes"}:
                need.append(pkg)

# dedupe
need = list(dict.fromkeys(need))
if need:
    print("Installiere:", ", ".join(need))
    _pip(*need)
else:
    print("Basis-Abhängigkeiten vorhanden (Embeddings/LLM optional).")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("IN_COLAB =", IN_COLAB)

In [ ]:
import os, re, json, math, hashlib
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False


# GPU-Hinweis früh (Colab ohne T4 → ungenaues kleines Modell)
try:
    import torch as _t0
    _has_cuda = _t0.cuda.is_available()
except Exception:
    _has_cuda = False
if not _has_cuda:
    _w = "Keine GPU! Laufzeit → Laufzeittyp ändern → T4-GPU, speichern, dann Alle ausführen. Sonst nur kleines Modell, ungenauer."
    try:
        from IPython.display import display, Markdown
        display(Markdown(f"> **⚠️ {_w}**"))
    except Exception:
        print("⚠️", _w)

HEADLESS = os.environ.get("COCKPIT_HEADLESS", "").strip().lower() in {"1", "true", "yes"}
if IN_COLAB:
    HEADLESS = False
SKIP_LLM = os.environ.get("SKIP_LLM", "0").strip().lower() in {"1", "true", "yes"}
if HEADLESS and "SKIP_LLM" not in os.environ:
    SKIP_LLM = True
SKIP_EMB = os.environ.get("SKIP_EMB", "1" if HEADLESS else "0").strip().lower() in {"1", "true", "yes"}
ALLOW_NET = os.environ.get("LOGOS_ALLOW_NET", "0" if HEADLESS else "1").strip().lower() in {"1", "true", "yes"}

if HEADLESS:
    matplotlib.use("Agg")

SEED = 42
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output, HTML
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
    class _Dummy:
        def __init__(self, **kw):
            self.__dict__.update(kw)
            self.value = kw.get("value", kw.get("description", False))
        def observe(self, *a, **k): pass
        def on_click(self, *a, **k): pass
    class widgets:
        Checkbox = ToggleButton = FloatSlider = Dropdown = Text = Button = Output = VBox = HBox = HTML = Label = Accordion = _Dummy
        Layout = lambda **kw: None
    def display(*a, **k):
        for x in a:
            print(x.to_string() if isinstance(x, (pd.DataFrame, pd.Series)) else x)
    def Markdown(s): return s
    def clear_output(*a, **k): pass
    def HTML(s): return s

if os.environ.get("COCKPIT_OUT"):
    OUT_DIR = Path(os.environ["COCKPIT_OUT"])
elif IN_COLAB:
    OUT_DIR = Path("/content/logos_cockpit_out")
else:
    OUT_DIR = Path("logos_cockpit_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Corpus-Ordner: neben Notebook / CWD / Colab-Upload
CANDIDATE_CORPUS = [
    Path("logos_corpus"),
    Path("k3/logos_corpus"),
    Path("/content/logos_corpus"),
    Path(__file__).resolve().parent / "logos_corpus" if "__file__" in dir() else Path("logos_corpus"),
]
# Notebook hat oft kein __file__; zusätzlich bekannte Smoke-Pfade
for p in [Path("/tmp/Bio2.0/k3/logos_corpus"), Path("/workspace/colab-artikel/cockpit/logos_corpus")]:
    CANDIDATE_CORPUS.append(p)

CORPUS_DIR = next((p for p in CANDIDATE_CORPUS if p.exists() and p.is_dir()), Path("logos_corpus"))
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

PUBLIC_BLOG = "https://rosary-mom.github.io/blog/2026-11-03-gottes-schoepferkraft-heute/"
print(f"SEED={SEED} | IN_COLAB={IN_COLAB} | HEADLESS={HEADLESS} | SKIP_LLM={SKIP_LLM} | SKIP_EMB={SKIP_EMB}")
print(f"OUT_DIR={OUT_DIR.resolve()} | CORPUS_DIR={CORPUS_DIR.resolve()}")

## 1 · Quellenregister (Provenienz)
Loader für `.md` / `.txt` und optional öffentliche URLs. **Corpus-Policy:** Eigene, von Uwe in Logos erstellte Inhalte (Notizen, Predigten, Auslegungen) sind zulässig per manuellem Export; Volltexte aus lizenzierten Logos-Bibliotheksressourcen nur als kurze Zitate mit Quellenangabe.

In [ ]:
BIBLE_REF_RE = re.compile(
    r"\b(?:"
    r"Ps(?:alm)?|Gen|Ex|Lev|Num|Dtn|Jos|Ri|Rut|1\s*Sam|2\s*Sam|1\s*Kön|2\s*Kön|"
    r"1\s*Chr|2\s*Chr|Esra|Neh|Est|Hiob|Spr|Pred|Hld|Jes|Jer|Klgl|Hes|Dan|"
    r"Hos|Joel|Am|Obad|Jona|Mi|Nah|Hab|Zef|Hag|Sach|Mal|"
    r"Mt|Mk|Lk|Joh|Apg|Röm|1\s*Kor|2\s*Kor|Gal|Eph|Phil|Kol|"
    r"1\s*Thess|2\s*Thess|1\s*Tim|2\s*Tim|Tit|Phlm|Heb|Jak|1\s*Petr|2\s*Petr|"
    r"1\s*Joh|2\s*Joh|3\s*Joh|Jud|Offb"
    r")\.?\s*\d+(?:\s*[,:]\s*\d+(?:\s*[–\-−]\s*\d+)?)?",
    re.I,
)

# "Vision": bewusst visionäre Texte (Phantasie), sichtbar, aber nicht am Score gemessen
RANK_ORDER = ["Schrift", "Tradition", "Forschung", "eigene Deutung", "Vision"]
RANK_WEIGHT = {"Schrift": 1.0, "Tradition": 0.85, "Forschung": 0.7, "eigene Deutung": 0.55, "Vision": 0.55}

def parse_frontmatter(text: str):
    meta = {}
    if text.startswith("---"):
        parts = text.split("---", 2)
        if len(parts) >= 3:
            for line in parts[1].splitlines():
                if ":" in line:
                    k, v = line.split(":", 1)
                    meta[k.strip()] = v.strip().strip('"')
            return meta, parts[2].lstrip("\n")
    return meta, text

def infer_rank(meta, text):
    r = (meta.get("rang") or meta.get("klasse") or "").lower()
    if "schrift" in r or "bibel" in r:
        return "Schrift"
    if "tradition" in r or "kirche" in r or "vater" in r:
        return "Tradition"
    if "vision" in r:
        return "Vision"
    if "forschung" in r or "research" in r:
        # Nur mit Nachweis (Diplomarbeit, UNFCCC o. ä.) als Forschung werten,
        # sonst bleibt es eigene Deutung (kein Aufschlag ohne Beleg).
        if (meta.get("beleg") or "").strip():
            return "Forschung"
        return "eigene Deutung"
    # Bibelblock-Heuristik
    if re.search(r"(Elberfelder|Luther|Einheitsübersetzung).{0,40}\d+", text) and len(BIBLE_REF_RE.findall(text)) >= 2:
        # eigener Blog mit Bibelzitaten bleibt "eigene Deutung", Zitate separat taggen
        pass
    return "eigene Deutung"

def chunk_text(text: str, max_chars=700, overlap=80):
    # an Absätzen, dann Fenster
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks = []
    buf = ""
    for p in paras:
        if len(buf) + len(p) + 2 <= max_chars:
            buf = (buf + "\n\n" + p).strip()
        else:
            if buf:
                chunks.append(buf)
            if len(p) <= max_chars:
                buf = p
            else:
                i = 0
                while i < len(p):
                    chunks.append(p[i:i+max_chars])
                    i += max_chars - overlap
                buf = ""
    if buf:
        chunks.append(buf)
    return chunks

def load_corpus_dir(corpus_dir: Path):
    docs = []
    for path in sorted(corpus_dir.glob("**/*")):
        if path.suffix.lower() not in {".md", ".txt"}:
            continue
        raw = path.read_text(encoding="utf-8", errors="replace")
        meta, body = parse_frontmatter(raw)
        rank = infer_rank(meta, body)
        docs.append({
            "doc_id": path.stem,
            "path": str(path),
            "title": meta.get("title") or path.stem,
            "source_url": meta.get("source_url") or "",
            "author": meta.get("author") or "",
            "date": meta.get("date") or "",
            "rank": rank,
            "klasse": meta.get("klasse") or rank,
            "corpus_policy": meta.get("corpus_policy") or "only_own_or_free",
            "beleg": meta.get("beleg") or "",
            "text": body,
            "sha1": hashlib.sha1(raw.encode("utf-8")).hexdigest()[:12],
        })
    return docs

DOCS = load_corpus_dir(CORPUS_DIR)
if not DOCS:
    # minimale Notiz, damit Notebook nicht leer abstürzt
    fallback = CORPUS_DIR / "README_CORPUS.md"
    fallback.write_text(
        "---\ntitle: \"Corpus leer\"\nrang: \"eigene Deutung\"\ncorpus_policy: \"only_own_or_free\"\n---\n\n"
        "Lege eigene/freie `.md`/`.txt` Dateien in logos_corpus/ ab.\n",
        encoding="utf-8",
    )
    DOCS = load_corpus_dir(CORPUS_DIR)

df_quellen = pd.DataFrame([{k: d[k] for k in ["doc_id","title","rank","klasse","beleg","source_url","author","date","sha1","corpus_policy","path"]} for d in DOCS])
display(df_quellen)
print("Dokumente:", len(DOCS), "| Policy: eigene Logos-Exporte ok; Library-Volltexte nur Kurzzitat+Quelle.")

## 2 · Evidenzleiter / Quellenrangfolge
**Schrift > Tradition > Forschung > eigene Deutung.** Jeder Chunk trägt Rang + Provenienz. Bibelstellen im Chunk werden erkannt und als Schrift-Bezüge gelistet (Kurzzitate bleiben Kurzzitate).

In [ ]:
import re as _re
HEADER_PREFIXES = ("**provenienz", "**öffentliche url", "**hinweis corpus")
def strip_header(text: str, title: str = "") -> str:
    """Kopfteil (Provenienz, URL, Corpus-Hinweis, Web-Boilerplate) vor dem Suchen ausblenden.
    Die Provenienz bleibt in den Metadaten (rank, source_url) erhalten."""
    out = []
    t = (title or "").strip().lower()
    for line in text.splitlines():
        l = line.strip().lower()
        if l.startswith(HEADER_PREFIXES):
            continue
        if "| rosary.health" in l or l in ("magnifica humanitas · bibel heute",):
            continue
        if t and l.lstrip("# ").strip() == t:
            continue  # doppelte Titelzeilen
        out.append(line)
    return _re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()

CHUNKS = []
for d in DOCS:
    for i, ch in enumerate(chunk_text(strip_header(d["text"], d["title"]))):
        refs = sorted(set(m.group(0) for m in BIBLE_REF_RE.finditer(ch)))
        # Wenn Chunk überwiegend Bibeltext-Block: höherer Schrift-Bezug, aber Doc-Rang bleibt Provenienz der Quelle
        CHUNKS.append({
            "chunk_id": f"{d['doc_id']}#{i}",
            "doc_id": d["doc_id"],
            "title": d["title"],
            "rank": d["rank"],
            "rank_w": RANK_WEIGHT.get(d["rank"], 0.5),
            "source_url": d["source_url"],
            "path": d["path"],
            "refs": refs,
            "text": ch,
        })

df_chunks = pd.DataFrame([{
    "chunk_id": c["chunk_id"], "title": c["title"], "rank": c["rank"],
    "refs": ", ".join(c["refs"][:6]), "zeichen": len(c["text"])
} for c in CHUNKS])
display(df_chunks.head(12))
print("Chunks:", len(CHUNKS))

# Evidenzleiter-Plot
fig, ax = plt.subplots(figsize=(7, 3), dpi=120)
ys = list(reversed(RANK_ORDER))
ws = [RANK_WEIGHT[r] for r in ys]
ax.barh(ys, ws, color=["#2ca02c", "#98df8a", "#ffbb78", "#fc8d62"])
ax.set_xlim(0, 1.05); ax.set_xlabel("Ranggewicht (Schema)")
ax.set_title("Evidenzleiter Theologie — methodisches Schema")
fig.tight_layout(); fig.savefig(OUT_DIR / "logos_evidenzleiter.png", bbox_inches="tight")
plt.show()

## 3 · Retrieval (BM25 lokal · Embeddings optional)
Standard: **reines Python-BM25** (ohne Netz). Optional: `intfloat/multilingual-e5-small`, wenn installiert und Netz erlaubt. Treffer zeigen Quelle + Rang.

In [ ]:
def tokenize(s: str):
    s = s.lower()
    s = re.sub(r"[^a-zäöüß0-9\s]", " ", s)
    return [t for t in s.split() if len(t) > 1]

class BM25:
    def __init__(self, docs_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = docs_tokens
        self.N = len(docs_tokens)
        self.df = Counter()
        self.doc_len = [len(d) for d in docs_tokens]
        self.avgdl = sum(self.doc_len) / max(1, self.N)
        for d in docs_tokens:
            for t in set(d):
                self.df[t] += 1
    def idf(self, t):
        n = self.df.get(t, 0)
        return math.log(1 + (self.N - n + 0.5) / (n + 0.5))
    def score(self, q_tokens, idx):
        d = self.docs[idx]; dl = self.doc_len[idx]; tf = Counter(d); s = 0.0
        for t in q_tokens:
            if t not in tf: continue
            s += self.idf(t) * (tf[t] * (self.k1 + 1)) / (tf[t] + self.k1 * (1 - self.b + self.b * dl / self.avgdl))
        return s

TOKENS = [tokenize(c["text"]) for c in CHUNKS]
BM = BM25(TOKENS)

EMB = None
EMB_MATRIX = None
if not SKIP_EMB and ALLOW_NET:
    try:
        from sentence_transformers import SentenceTransformer
        EMB = SentenceTransformer("intfloat/multilingual-e5-small")
        EMB_MATRIX = EMB.encode(["passage: " + c["text"] for c in CHUNKS], normalize_embeddings=True)
        print("Embeddings aktiv: intfloat/multilingual-e5-small")
    except Exception as e:
        print("Embeddings nicht verfügbar → BM25:", e)
else:
    print("Retrieval: BM25 (SKIP_EMB/ALLOW_NET) — offline-fähig.")

def search(query: str, top_k=5):
    q_tok = tokenize(query)
    scores = []
    if EMB is not None and EMB_MATRIX is not None:
        qv = EMB.encode(["query: " + query], normalize_embeddings=True)[0]
        sims = EMB_MATRIX @ qv
        for i, sim in enumerate(sims):
            # Ranggewicht multiplikativ
            scores.append((float(sim) * CHUNKS[i]["rank_w"], i, "emb"))
    else:
        for i in range(len(CHUNKS)):
            scores.append((BM.score(q_tok, i) * CHUNKS[i]["rank_w"], i, "bm25"))
    scores.sort(reverse=True)
    hits = []
    for sc, i, mode in scores[:top_k]:
        c = CHUNKS[i]
        hits.append({
            "score": round(sc, 4),
            "mode": mode,
            "chunk_id": c["chunk_id"],
            "title": c["title"],
            "rank": c["rank"],
            "refs": c["refs"],
            "source_url": c["source_url"],
            "text": c["text"][:500] + ("…" if len(c["text"]) > 500 else ""),
            "text_full": c["text"],
        })
    return hits

DEMO_Q = "Was sagt Ps 139 über Gottes Schöpferkraft heute?"
HITS = search(DEMO_Q, top_k=5)
df_hits = pd.DataFrame([{k: v for k, v in h.items() if k != "text_full"} for h in HITS])
display(Markdown(f"**Demo-Frage:** {DEMO_Q}"))
display(df_hits)
STATE = {"demo_q": DEMO_Q, "hits": HITS}
if HITS:
    print("Top-Hit:", HITS[0]["chunk_id"], "|", HITS[0]["rank"], "| score=", HITS[0]["score"])
    print(HITS[0]["text"][:280])

## 4 · Drei Achsen (Heuristik)
Gewichte für **Exegese** (textnah), **Lehre** (stimmig), **Praxis** (Seelsorge/Gemeinde). Score ist **transparent heuristisch**, keine Lehrentscheidung.

In [ ]:
def axis_scores(hit, w_ex=1/3, w_le=1/3, w_pr=1/3):
    text = hit["text_full"].lower()
    refs_n = len(hit["refs"])
    # Exegese: Bibelstellen + Nähe zu „psalm/text/vers“
    ex = min(1.0, 0.25 * refs_n + (0.35 if re.search(r"psalm|vers|bibeltext|elberfelder", text) else 0) + 0.2 * min(1, hit["score"]/5))
    # Lehre: Schlüssel theologisch
    le = 0.2
    for kw in ["schöpf", "geschaff", "theolog", "lehre", "zweck", "ehre", "gnade",
               "christus", "werk", "gottes gedanken", "ewigkeit", "töpfer"]:
        if kw in text: le += 0.1
    le = min(1.0, le)
    # Praxis: Seelsorge/Gemeinde
    pr = 0.15
    for kw in ["impuls", "heute", "dank", "bet", "gebet", "gemeinde", "schritt", "seelsorge",
               "wandeln", "gute werke", "dein leben", "du bist", "erfüllung", "neuer tag"]:
        if kw in text: pr += 0.1
    pr = min(1.0, pr)
    # Rangpriorität
    rw = hit.get("rank_w") or RANK_WEIGHT.get(hit["rank"], 0.5)
    # hit may not have rank_w
    if "rank_w" not in hit:
        rw = RANK_WEIGHT.get(hit["rank"], 0.5)
    total = (w_ex*ex + w_le*le + w_pr*pr) * (0.7 + 0.3*rw)
    return {"exegese": round(ex,3), "lehre": round(le,3), "praxis": round(pr,3), "S_axis": round(total,3), "rang_w": rw}

def score_answer(hits, w_ex=1/3, w_le=1/3, w_pr=1/3):
    wsum = w_ex + w_le + w_pr
    w_ex, w_le, w_pr = w_ex/wsum, w_le/wsum, w_pr/wsum
    rows = []
    for h in hits:
        h2 = dict(h)
        h2["rank_w"] = RANK_WEIGHT.get(h["rank"], 0.5)
        ax = axis_scores(h2, w_ex, w_le, w_pr)
        rows.append({**ax, "chunk_id": h["chunk_id"], "rank": h["rank"], "retrieval": h["score"]})
    df = pd.DataFrame(rows)
    S = float(df["S_axis"].mean()) if len(df) else 0.0
    return df, round(S, 4), {"w_ex": w_ex, "w_le": w_le, "w_pr": w_pr}

out_ax = widgets.Output() if HAS_WIDGETS else None
w_ex = widgets.FloatSlider(value=0.40, min=0, max=1, step=0.05, description="Exegese") if HAS_WIDGETS else None
w_le = widgets.FloatSlider(value=0.30, min=0, max=1, step=0.05, description="Lehre") if HAS_WIDGETS else None
w_pr = widgets.FloatSlider(value=0.30, min=0, max=1, step=0.05, description="Praxis") if HAS_WIDGETS else None

def render_axes(ex=0.4, le=0.3, pr=0.3):
    df, S, w = score_answer(STATE.get("hits") or [], ex, le, pr)
    STATE["axis_df"] = df.to_dict(orient="records")
    STATE["S_axis"] = S
    STATE["weights"] = w
    msg = f"**Heuristik-Score S_axis={S}** mit Gewichten {w} — Kennzeichnung: Heuristik, keine Lehramt-Aussage."
    if HAS_WIDGETS and out_ax is not None:
        with out_ax:
            clear_output(wait=True)
            display(Markdown(msg)); display(df)
    else:
        print(msg.replace("**",""))
        print(df.to_string(index=False) if len(df) else "(keine Hits)")
    fig, ax = plt.subplots(figsize=(6,3), dpi=120)
    if len(df):
        ax.bar(["Exegese","Lehre","Praxis","S_axis"],
               [df["exegese"].mean(), df["lehre"].mean(), df["praxis"].mean(), S],
               color=["#2ca02c","#1f77b4","#ff7f0e","#111"])
    ax.set_ylim(0,1.05); ax.set_title("Theologische Achsen (Heuristik)")
    fig.tight_layout(); fig.savefig(OUT_DIR/"logos_achsen.png", bbox_inches="tight")
    if HEADLESS: plt.close(fig)
    else: plt.show()
    return S

if HAS_WIDGETS and not HEADLESS:
    def _chg(_=None):
        render_axes(w_ex.value, w_le.value, w_pr.value)
    for w in (w_ex, w_le, w_pr):
        w.observe(_chg, names="value")
    display(widgets.HBox([w_ex, w_le, w_pr]), out_ax)
    _chg()
else:
    render_axes()

## 5 · Gates / Maßgaben
1. **Kein Beleg → keine Aussage**  
2. **Deutung als Deutung markieren**  
3. **Bibelstellen erkennen** (deutsche Referenzen)

In [ ]:
# Bestehensschwelle wie bei den Moodle-Selbstkontroll-Tests (67,0 %)
PASS_THRESHOLD = 0.67

def gates_for(hits, claim_is_deutung=True):
    belege = []
    refs = []
    for h in hits:
        belege.append(h["chunk_id"])
        refs.extend(h["refs"])
    refs = sorted(set(refs))
    g = {
        "Beleg vorhanden": len(hits) > 0,
        "Quellen angegeben": all(bool(h.get("source_url") or h.get("title")) for h in hits) if hits else False,
        "Deutung markiert": bool(claim_is_deutung),
        "Bibelstelle erkannt": len(refs) > 0,
    }
    # Vision-Texte: Phantasie bleibt sichtbar, wird aber nicht am Score gemessen
    is_vision = bool(hits) and all(h.get("rank") == "Vision" for h in hits)
    if not is_vision:
        g[f"Selbstkontrolle ≥ {PASS_THRESHOLD:.0%}"] = float(STATE.get("S_axis") or 0) >= PASS_THRESHOLD
    if is_vision:
        status = "VISION – nicht bewertet (als Phantasie kennzeichnen)"
    else:
        status = "FREIGABE MÖGLICH (menschlich prüfen)" if all(g.values()) else "REFINEMENT REQUIRED"
    return {"gates": g, "status": status, "refs": refs, "belege": belege}

GATES = gates_for(STATE.get("hits") or [], claim_is_deutung=True)
STATE["gates"] = GATES
display(pd.DataFrame([(k, "OK" if v else "FEHLT") for k,v in GATES["gates"].items()], columns=["Gate","Status"]))
print("Status:", GATES["status"])
print("Bibelstellen:", ", ".join(GATES["refs"][:12]) or "—")

## 6 · LLM-Assistent (optional, nur aus Treffern)
Qwen2.5-7B-Instruct 4-Bit auf T4, Fallback 1.5B. **Antwortet nur mit den gebrachten Chunks + Zitaten.** Priorität: theologische Forschung für LOGOS Academical Research Library / UniCampus. Kein API-Key.

In [8]:
LLM = {"ready": False, "name": None, "pipe": None, "error": None}

GLOSSAR = {
    "Evidenzleiter": "Quellenrangfolge Schrift > Tradition > Forschung > eigene Deutung (Schema mit Ranggewichten).",
    "Corpus_Policy": "Eigene, von Uwe in Logos erstellte Inhalte (Notizen, Predigten, Auslegungen) sind zulässig per manuellem Export; Volltexte aus lizenzierten Logos-Bibliotheksressourcen nur als kurze Zitate mit Quellenangabe.",
    "Gates": "Kein Beleg → keine Aussage; Quellen angegeben; Deutung markiert; Bibelstelle erkannt (deutsche Referenzen).",
    "REFINEMENT_REQUIRED": "Mindestens ein Gate fehlt.",
    "FREIGABE_MOEGLICH": "Alle Gates OK — trotzdem menschlich prüfen. Keine Lehramt-Entscheidung.",
    "S_axis": "Heuristik-Score aus Achsen Exegese/Lehre/Praxis × Ranggewicht. Kennzeichnung: Heuristik, keine Lehraussage.",
    "Retrieval": "BM25 lokal; optional multilingual-e5-small. Antworten nur aus Treffer-Chunks.",
    "Seestern": "Logos Max bleibt das menschliche Werkzeug; Cockpit nutzt nur erlaubtes Corpus.",
}

def _glossar_text():
    return "GLOSSAR (verbindlich):\n" + "\n".join(f"- {k}: {v}" for k, v in GLOSSAR.items())

SYSTEM = f"""Du bist Assistent für die LOGOS Academical Research Library / UniCampus (ROSARY, Magnifica Humanitas).
Priorität: theologische Forschung (Exegese, Lehre, Praxis/Seelsorge).

HARTREGELN:
1) Nutze NUR bereitgestellte Corpus-Treffer + GLOSSAR. Kein Weltwissen, nichts erfinden.
2) Fehlt der Beleg: antworte genau „Steht nicht im Cockpit.“
3) Kennzeichne Deutung als Deutung; nenne chunk_id und Bibelstellen.
4) Zitiere, welche Treffer/Werte du nutzt.

{_glossar_text()}
"""

def build_context(hits):
    blocks = []
    for h in hits:
        blocks.append(
            f"[{h['chunk_id']} | Rang={h['rank']} | Refs={', '.join(h['refs']) or '—'}]\n{h['text_full']}"
        )
    return "\n\n".join(blocks)

def _warn_no_gpu(use_cuda: bool):
    if use_cuda:
        return
    msg = "Keine GPU! Laufzeit → Laufzeittyp ändern → T4-GPU, speichern, dann Alle ausführen. Sonst nur kleines Modell, ungenauer."
    try:
        from IPython.display import display, Markdown
        display(Markdown(f"> **⚠️ {msg}**"))
    except Exception:
        print("⚠️", msg)

def load_llm():
    if SKIP_LLM:
        LLM.update({"ready": False, "name": "STUB", "error": "SKIP_LLM=1 — Stub aktiv."})
        print(LLM["error"]); return False
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
    except Exception as e:
        LLM["error"] = f"transformers/torch fehlt: {e}"; print(LLM["error"]); return False
    use_cuda = torch.cuda.is_available()
    _warn_no_gpu(use_cuda)
    print("CUDA:", use_cuda, (torch.cuda.get_device_name(0) if use_cuda else ""))
    cands = []
    if use_cuda:
        cands.append(("Qwen/Qwen2.5-7B-Instruct", True))
    cands.append(("Qwen/Qwen2.5-1.5B-Instruct", False))
    last = None
    for name, bit4 in cands:
        try:
            print(f"Lade {name} …")
            tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
            kw = {"trust_remote_code": True}
            if bit4 and use_cuda:
                from transformers import BitsAndBytesConfig
                kw["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
                )
                kw["device_map"] = "auto"
            else:
                kw["dtype"] = torch.float16 if use_cuda else torch.float32
                if use_cuda: kw["device_map"] = "auto"
            model = AutoModelForCausalLM.from_pretrained(name, **kw)
            gen = pipeline("text-generation", model=model, tokenizer=tok)
            LLM.update({"ready": True, "name": name, "pipe": gen, "error": None})
            print("LLM bereit:", name); return True
        except Exception as e:
            last = e; print("Fail", name, e)
            try:
                import torch as _t
                if use_cuda: _t.cuda.empty_cache()
            except Exception:
                pass
    LLM["error"] = f"Kein LLM: {last}"; print(LLM["error"]); return False

def ask_logos(frage: str, top_k=4, max_new_tokens=200):
    hits = search(frage, top_k=top_k)
    STATE["last_q"] = frage; STATE["hits"] = hits
    g = gates_for(hits, True)
    if not g["gates"]["Beleg vorhanden"]:
        return "Steht nicht im Cockpit."
    ctx = build_context(hits)
    if not LLM["ready"]:
        top = hits[0]
        return (
            f"[Stub — {LLM.get('error')}]\n"
            f"Frage: {frage}\n"
            f"Beleg {top['chunk_id']} (Rang {top['rank']}): {top['text'][:320]}\n"
            f"Bibelstellen: {', '.join(g['refs']) or '—'}\n"
            f"Status: {g['status']} | Deutung als Deutung. Genutzte Chunks: {[h['chunk_id'] for h in hits]}."
        )
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Treffer:\n{ctx}\n\nFrage: {frage}\nAntworte nur aus den Treffern + Glossar."},
    ]
    try:
        prompt = LLM["pipe"].tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt = SYSTEM + "\n\n" + ctx + "\n\nFrage: " + frage + "\nAssistent:"
    out = LLM["pipe"](
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.1,
        return_full_text=False,
    )[0]["generated_text"]
    text = out.strip()
    if "Assistent:" in text:
        text = text.split("Assistent:")[-1].strip()
    return text

try:
    import torch as _torch_chk
    _warn_no_gpu(_torch_chk.cuda.is_available())
except Exception:
    _warn_no_gpu(False)

load_llm()
ANS = ask_logos(DEMO_Q)
print(ANS)
STATE["answer"] = ANS

if HAS_WIDGETS and not HEADLESS:
    qbox = widgets.Text(value=DEMO_Q, description="Frage:", layout=widgets.Layout(width="75%"))
    btn = widgets.Button(description="Fragen")
    out_q = widgets.Output()
    def _go(_):
        with out_q:
            clear_output(wait=True)
            print(ask_logos(qbox.value))
    btn.on_click(_go)
    display(widgets.HBox([qbox, btn]), out_q)


CUDA: True Tesla T4
Lade Qwen/Qwen2.5-7B-Instruct …


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

LLM bereit: Qwen/Qwen2.5-7B-Instruct
Steht nicht im Cockpit.


Output()

## 7 · Export JSON

In [9]:
export = {
    "projekt": "LOGOS-Cockpit / UniCampus",
    "seed": SEED,
    "corpus_dir": str(CORPUS_DIR),
    "corpus_policy": "own_logos_exports_ok; licensed_library_fulltext_only_short_quotes_with_citation",
    "docs": df_quellen.to_dict(orient="records"),
    "demo_q": STATE.get("demo_q"),
    "hits": [{k:v for k,v in h.items() if k != "text_full"} for h in STATE.get("hits") or []],
    "S_axis": STATE.get("S_axis"),
    "weights": STATE.get("weights"),
    "gates": STATE.get("gates"),
    "answer_preview": (STATE.get("answer") or "")[:1200],
    "llm": {"ready": LLM["ready"], "name": LLM["name"], "error": LLM["error"]},
    "leitsatz": "Verstehen schlägt bloße Größe",
    "seestern": "Logos Max bleibt das menschliche Werkzeug; Cockpit nur eigenes/freies Material",
}

def _jsonable(o):
    if isinstance(o, dict):
        return {k: _jsonable(v) for k,v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    if isinstance(o, (np.floating, float)):
        return float(o)
    if isinstance(o, (np.integer, int)):
        return int(o)
    return o

path = OUT_DIR / "logos_cockpit_state.json"
path.write_text(json.dumps(_jsonable(export), ensure_ascii=False, indent=2), encoding="utf-8")
print("geschrieben:", path)
# Top-Hit für Smoke-Report
if STATE.get("hits"):
    print("SMOKE_TOP_HIT:", STATE["hits"][0]["chunk_id"], STATE["hits"][0]["score"], STATE["hits"][0]["text"][:160].replace("\n"," "))

geschrieben: /content/logos_cockpit_out/logos_cockpit_state.json
SMOKE_TOP_HIT: README_CORPUS#0 0.4251 Lege eigene/freie `.md`/`.txt` Dateien in logos_corpus/ ab.


---
**Corpus-Hinweis:** Eigene, von Uwe in Logos erstellte Inhalte (Notizen, Predigten, Auslegungen) sind zulässig per manuellem Export; Volltexte aus lizenzierten Logos-Bibliotheksressourcen nur als kurze Zitate mit Quellenangabe. `[1]–[4]` im Blog = Zitationen.  
Öffentliche Demo-Quelle: https://rosary-mom.github.io/blog/2026-11-03-gottes-schoepferkraft-heute/